# Savant 04 guided base model after the error fixes

This notebook measures the base model again after three changes that target its failed code steps. Date columns now reach the model's code as pandas datetimes, the prompt says that every step starts fresh in a new process, and the most common errors come back with a one-line hint.

It answers the same 30 guided bench tables as the second fine-tune pilot and the ten blind baseline tables, so both numbers compare directly with earlier runs. No training happens here.


In [ ]:
import os

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import glob
import json
import subprocess
import sys
import time
from pathlib import Path

build = sorted(glob.glob("/kaggle/input/**/BUILD", recursive=True))
assert build and Path(build[0]).read_text().strip() == "EXPECTED_BUILD", "stale bench dataset, stop"
wheel = sorted(glob.glob("/kaggle/input/**/savant-*.whl", recursive=True))[-1]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", wheel], check=True)
import torch  # noqa: E402

TASKS = sorted(glob.glob("/kaggle/input/**/tasks/*.csv", recursive=True))
print(wheel, len(TASKS), "tables")
print([torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

In [ ]:
# Settings for this run.
MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"
GUIDED_TABLES = {"null": 5, "hard-null": 5, "plant": 20}  # the same tables as 03-finetune
BLIND_TABLES = {"null": 3, "hard-null": 3, "plant": 4}  # the ten baseline tables
MAX_NEW_TOKENS = 768
KEEP_MESSAGES = 6

In [ ]:
import polars as pl

from savant.agent import ask, parse_pairs
from savant.grade import derived_pairs
from savant.llm import Transformers
from savant.planted import Plant, Task
from savant.traces import PAIR_QUESTION


def load_task(csv):
    truth = json.loads(Path(csv.replace(".csv", ".truth.json")).read_text())
    plants = [Plant(p["kind"], tuple(p["columns"]), p["strength"]) for p in truth["plants"]]
    return Task(truth["id"], pl.read_csv(csv, try_parse_dates=True), plants)


tasks = [load_task(c) for c in TASKS]


def pick(counts):
    out = []
    for prefix, count in counts.items():
        out += [t for t in tasks if t.id.startswith(prefix + "-")][:count]
    return out


log = Path("/kaggle/working/runs.jsonl")
log.write_text("")
runs = []


def evaluate(client, model_label, mode, chosen):
    for t in chosen:
        t0 = time.time()
        try:
            r = ask(
                t.table,
                PAIR_QUESTION,
                client,
                keep_messages=KEEP_MESSAGES,
                show_findings=(mode == "guided"),
            )
            error = ""
        except torch.cuda.OutOfMemoryError as e:
            r, error = None, f"out of memory: {e}"[:300]
        torch.cuda.empty_cache()
        real = {frozenset(p.columns) for p in t.plants if p.kind != "leak"}
        leaks = {frozenset(p.columns) for p in t.plants if p.kind == "leak"}
        claimed = parse_pairs(r.rendered if r else "", t.table.columns)
        row = {
            "model": model_label,
            "mode": mode,
            "task": t.id,
            "kind": t.id.rsplit("-", 1)[0],
            "grounded": bool(r and r.grounded),
            "rounds": r.rounds if r else None,
            "steps": len(r.steps) if r else 0,
            "failed_steps": sum(not s.ok for s in r.steps) if r else 0,
            "planted": len(real),
            "found": len(claimed & real),
            "false": len(claimed - real - leaks - derived_pairs(t)),
            "unverified": len(r.unverified_pairs) if r else 0,
            "seconds": round(time.time() - t0, 1),
            "error": error,
            "answer": r.rendered if r else None,
            "transcript": r.transcript if r else None,
        }
        runs.append(row)
        with log.open("a") as f:
            f.write(json.dumps(row, default=str) + "\n")
        print(
            model_label,
            mode,
            t.id,
            "grounded" if row["grounded"] else "NOT grounded",
            row["found"],
            "of",
            row["planted"],
            "found",
            row["false"],
            "false",
            row["seconds"],
            "s",
            error,
            flush=True,
        )


client = Transformers(MODEL_ID, max_new_tokens=MAX_NEW_TOKENS)
evaluate(client, "base", "guided", pick(GUIDED_TABLES))
evaluate(client, "base", "blind", pick(BLIND_TABLES))

In [ ]:
def share(xs):
    return round(sum(xs) / len(xs), 3) if xs else None


agent_summary = {"model": MODEL_ID, "tables": {"guided": GUIDED_TABLES, "blind": BLIND_TABLES}}
for label, mode in [("base", "guided"), ("base", "blind")]:
    rs = [r for r in runs if r["model"] == label and r["mode"] == mode]
    planted = [r for r in rs if r["kind"] == "plant"]
    agent_summary[f"{label} {mode}"] = {
        "tables": len(rs),
        "grounded_answers": share([r["grounded"] for r in rs]),
        "mean_rounds": share([r["rounds"] for r in rs if r["rounds"]]),
        "mean_seconds": share([r["seconds"] for r in rs]),
        "failed_step_share": round(
            sum(r["failed_steps"] for r in rs) / max(1, sum(r["steps"] for r in rs)), 3
        ),
        "planted_pairs": sum(r["planted"] for r in planted),
        "recall": round(
            sum(r["found"] for r in planted) / max(1, sum(r["planted"] for r in planted)), 3
        ),
        "false_per_null_table": share([r["false"] for r in rs if r["kind"] == "null"]),
        "false_per_hard_null_table": share([r["false"] for r in rs if r["kind"] == "hard-null"]),
        "false_per_planted_table": share([r["false"] for r in planted]),
        "unverified_claims": sum(r["unverified"] for r in rs),
        "errors": sum(bool(r["error"]) for r in rs),
    }
Path("/kaggle/working/guided.json").write_text(
    json.dumps(
        {"agent": agent_summary},
        indent=1,
        default=str,
    )
)
print(json.dumps(agent_summary, indent=1))